In [1]:
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_ENTRADA = "CIC17__cleanning__v1"
NOMBRE_SPLIT = "CIC17__split__v1"

RUTA_DATASET_LIMPIO = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_ENTRADA
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_ENTRADA}.csv"
NOMBRE_TRAIN = f"{NOMBRE_SPLIT}__train.csv"
NOMBRE_TEST = f"{NOMBRE_SPLIT}__test.csv"
NOMBRE_REPORTE = f"{NOMBRE_SPLIT}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [3]:
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()

print("Dataset limpio de entrada:")
print(RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO)
print()

print("Ruta de salida:")
print(RUTA_SALIDA)

PROJECT_ROOT:
/LUSTRE/home/inginf/u32902122/TFG

Dataset limpio de entrada:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC17__cleanning__v1/CIC17__cleanning__v1.csv

Ruta de salida:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC17__split__v1


In [4]:
input_path = RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO

if not input_path.exists():
    raise FileNotFoundError(f"No existe el dataset limpio en: {input_path}")

df = cargar_dataset(nombre_dataset=NOMBRE_DATASET_LIMPIO, ruta_base=RUTA_DATASET_LIMPIO)

shape_original = df.shape

print("Forma del dataset limpio:")
print(shape_original)

Forma del dataset limpio:
(2520798, 48)


In [5]:
df.head()

,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,54865,3,2,12,6,6,6.0,0,0,4.000000e+06,...,33,-1,1,20,0.0,0.0,0,0,0.0,BENIGN
1,55054,109,1,6,6,6,6.0,6,6,1.100917e+05,...,29,256,0,20,0.0,0.0,0,0,0.0,BENIGN
2,55055,52,1,6,6,6,6.0,6,6,2.307692e+05,...,29,256,0,20,0.0,0.0,0,0,0.0,BENIGN
3,46236,34,1,6,6,6,6.0,6,6,3.529412e+05,...,31,329,0,20,0.0,0.0,0,0,0.0,BENIGN
4,54863,3,2,12,6,6,6.0,0,0,4.000000e+06,...,32,-1,1,20,0.0,0.0,0,0,0.0,BENIGN


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución global de clases:")
display(resumen_clases(df, LABEL_COL))

Columna objetivo encontrada correctamente.

Distribución global de clases:


,count,percentage
LABEL,,
BENIGN,2095057,83.1109
Bot,1948,0.0773
DDoS,128014,5.0783
DoS GoldenEye,10286,0.4080
DoS Hulk,172846,6.8568
DoS Slowhttptest,5228,0.2074
DoS slowloris,5385,0.2136
FTP-Patator,5931,0.2353
Heartbleed,11,0.0004


In [7]:
train_df, test_df = dividir_train_test_stratified(
    df=df,
    label_col=LABEL_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Forma train:", train_df.shape)
print("Forma test:", test_df.shape)

Forma train: (2016638, 48)
Forma test: (504160, 48)


In [8]:
print("Distribución de clases en TRAIN:")
display(resumen_clases(train_df, LABEL_COL))

Distribución de clases en TRAIN:


,count,percentage
LABEL,,
BENIGN,1676045,83.1109
Bot,1558,0.0773
DDoS,102411,5.0783
DoS GoldenEye,8229,0.4081
DoS Hulk,138277,6.8568
DoS Slowhttptest,4182,0.2074
DoS slowloris,4308,0.2136
FTP-Patator,4745,0.2353
Heartbleed,9,0.0004


In [9]:
print("Distribución de clases en TEST:")
display(resumen_clases(test_df, LABEL_COL))

Distribución de clases en TEST:


,count,percentage
LABEL,,
BENIGN,419012,83.1109
Bot,390,0.0774
DDoS,25603,5.0783
DoS GoldenEye,2057,0.4080
DoS Hulk,34569,6.8568
DoS Slowhttptest,1046,0.2075
DoS slowloris,1077,0.2136
FTP-Patator,1186,0.2352
Heartbleed,2,0.0004


In [10]:
resumen_global = resumen_clases(df, LABEL_COL).rename(
    columns={"count": "global_count", "percentage": "global_percentage"}
)

resumen_train = resumen_clases(train_df, LABEL_COL).rename(
    columns={"count": "train_count", "percentage": "train_percentage"}
)

resumen_test = resumen_clases(test_df, LABEL_COL).rename(
    columns={"count": "test_count", "percentage": "test_percentage"}
)

comparacion = pd.concat([resumen_global, resumen_train, resumen_test], axis=1)

print("Comparación global / train / test:")
display(comparacion)

Comparación global / train / test:


,global_count,global_percentage,train_count,train_percentage,test_count,test_percentage
LABEL,,,,,,
BENIGN,2095057,83.1109,1676045,83.1109,419012,83.1109
Bot,1948,0.0773,1558,0.0773,390,0.0774
DDoS,128014,5.0783,102411,5.0783,25603,5.0783
DoS GoldenEye,10286,0.4080,8229,0.4081,2057,0.4080
DoS Hulk,172846,6.8568,138277,6.8568,34569,6.8568
DoS Slowhttptest,5228,0.2074,4182,0.2074,1046,0.2075
DoS slowloris,5385,0.2136,4308,0.2136,1077,0.2136
FTP-Patator,5931,0.2353,4745,0.2353,1186,0.2352
Heartbleed,11,0.0004,9,0.0004,2,0.0004


In [11]:
guardar_dataset_csv(
    df=train_df,
    nombre_archivo=NOMBRE_TRAIN,
    ruta=RUTA_SALIDA
)

guardar_dataset_csv(
    df=test_df,
    nombre_archivo=NOMBRE_TEST,
    ruta=RUTA_SALIDA
)

print("Train guardado en:")
print(RUTA_SALIDA / NOMBRE_TRAIN)
print()
print("Test guardado en:")
print(RUTA_SALIDA / NOMBRE_TEST)

Train guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC17__split__v1/CIC17__split__v1__train.csv

Test guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC17__split__v1/CIC17__split__v1__test.csv


In [12]:
reporte_split = {
    "dataset_entrada": NOMBRE_DATASET_LIMPIO,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "shape_global": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "shape_train": {
        "rows": int(train_df.shape[0]),
        "cols": int(train_df.shape[1])
    },
    "shape_test": {
        "rows": int(test_df.shape[0]),
        "cols": int(test_df.shape[1])
    },
    "class_distribution_global": {
        str(k): int(v) for k, v in df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_train": {
        str(k): int(v) for k, v in train_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_test": {
        str(k): int(v) for k, v in test_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    }
}

report_path = RUTA_SALIDA / NOMBRE_REPORTE

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(reporte_split, f, indent=2, ensure_ascii=False)

print("Reporte guardado en:")
print(report_path)

Reporte guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC17__split__v1/CIC17__split__v1_report.json


In [13]:
print("========== RESUMEN SPLIT ==========")
print(f"Dataset de entrada: {NOMBRE_DATASET_LIMPIO}")
print(f"Forma global: {df.shape}")
print(f"Forma train: {train_df.shape}")
print(f"Forma test: {test_df.shape}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print("===================================")

========== RESUMEN SPLIT ==========
Dataset de entrada: CIC17__cleanning__v1.csv
Forma global: (2520798, 48)
Forma train: (2016638, 48)
Forma test: (504160, 48)
Test size: 0.2
Random state: 42
